# RRF (Reciprocal Rank Fusion — объединение по обратным рангам)

Забудем про оценки вообще, используем только позиции (ранги). Каждый метод голосует за документ, в зависимости только от того, на каком месте он его поставил:
вклад = 1 / (k + позиция)

Документ на 1-м месте даёт большой голос (1/(k+1)), на 50-м - маленький (1/(k+50)). Складываем голоса от BM25 и от LSA и сортируем по сумме. Что получается: статья, которую оба метода поставили высоко, набирает много и всплывает наверх. Статья, которую один поднял, а другой утопил, получает средне. А k (обычно 10–60) — это смягчитель, чтобы разница между 1-м и 2-м местом не была слишком критичной

In [1]:
import pandas as pd, numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from metrics import map_at_k
from text_processing import clean_html, tokenize

In [3]:
arts = pd.read_feather('../candidate_public/candidate_data/articles.f')
cal = pd.read_feather('../candidate_public/candidate_data/calibration.f')
gt = {r.query_id:{int(x) for x in r.ground_truth.split()} for r in cal.itertuples()}
ids = arts.article_id.tolist()
doc = (arts.title + ' ') * 3 + arts.body.map(clean_html)

tok0 = lambda s: tokenize(s, stem=False)
bm = BM25Okapi([tok0(t) for t in doc])
S_bm = np.vstack([bm.get_scores(tok0(q)) for q in cal.query_text])

tok = lambda s: tokenize(s, stem=True)
vec = TfidfVectorizer(analyzer=tok, sublinear_tf=True, min_df=2)
A = vec.fit_transform(doc)
svd = TruncatedSVD(n_components=100, random_state=42)
At = normalize(svd.fit_transform(A))
S_ls = cosine_similarity(normalize(svd.transform(vec.transform(cal.query_text))), At)

def ranks(row):
    order = np.argsort(row)[::-1]; r = np.empty(len(row), int); r[order]=np.arange(len(row)); return r
Rb = np.vstack([ranks(S_bm[i]) for i in range(len(cal))])
Rl = np.vstack([ranks(S_ls[i]) for i in range(len(cal))])

def topk(S): return {r.query_id:[ids[i] for i in np.argsort(S[qi])[::-1][:10]] for qi,r in enumerate(cal.itertuples())}
print(f'BM25 один: MAP@10 = {map_at_k(topk(S_bm), gt):.4f}')
print(f'LSA один: MAP@10 = {map_at_k(topk(S_ls), gt):.4f}')
for wbm in (0.5, 0.6, 0.7):
    fused = wbm/(10+Rb) + (1-wbm)/(10+Rl)
    print(f'RRF(BM25 w={wbm}, LSA): MAP@10 = {map_at_k(topk(fused), gt):.4f}')

/home/d_tsyp/projects/support-article-ranking/.venv/lib/python3.14/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


BM25 один: MAP@10 = 0.3025
LSA один: MAP@10 = 0.2842
RRF(BM25 w=0.5, LSA): MAP@10 = 0.3639
RRF(BM25 w=0.6, LSA): MAP@10 = 0.3582
RRF(BM25 w=0.7, LSA): MAP@10 = 0.3419
